# 発話の意図を分類する (少数の例から)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/21_dialogue/intent_classifier.ipynb)

NLP クックブック 21-03 ／ 対話システム

ユーザーの発話が「予約したい」「キャンセルしたい」「営業時間を知りたい」のどれなのかを判定したい。意図ごとに数件の例文しかない。

In [1]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression

examples = {
    "予約": ["予約をお願いします", "席を取りたいのですが", "明日の夜 2 名で予約できますか", "来週の金曜に予約したい", "テーブルを押さえてほしい"],
    "キャンセル": ["予約をキャンセルしたい", "予約を取り消してください", "行けなくなったのでキャンセルで", "予約をやめたいです", "キャンセルをお願いします"],
    "変更": ["予約の時間を変更したい", "人数を 3 人に変えてください", "日にちをずらせますか", "予約を 19 時に変更できますか", "予約内容を変えたい"],
    "営業時間": ["何時まで開いていますか", "営業時間を教えてください", "日曜日はやっていますか", "定休日はいつですか", "ランチは何時からですか"],
    "場所": ["お店はどこにありますか", "最寄り駅を教えて", "駐車場はありますか", "住所を知りたい", "駅から歩いて何分ですか"],
    "雑談": ["こんにちは", "ありがとう", "おすすめの映画は？", "今日は寒いですね", "あなたは誰ですか"],
}
model = SentenceTransformer("intfloat/multilingual-e5-small")
texts = [t for ts in examples.values() for t in ts]
labels = [intent for intent, ts in examples.items() for _ in ts]
clf = LogisticRegression(C=1000, max_iter=5000).fit(model.encode(["query: " + t for t in texts], normalize_embeddings=True), labels)

def classify(text, min_prob=0.55):
    probs = clf.predict_proba(model.encode(["query: " + text], normalize_embeddings=True))[0]
    order = np.argsort(-probs)
    intent = str(clf.classes_[order[0]]) if probs[order[0]] >= min_prob else "不明"
    return intent, [(str(clf.classes_[i]), round(float(probs[i]), 2)) for i in order[:2]]

for text in ["土曜の 18 時に 4 人で行きたいんですが", "やっぱり行けなくなりました", "予約を 1 時間遅らせたい", "月曜って開いてる？",
             "車で行けますか", "予約", "領収書は出ますか？", "今日の夕飯は何にしよう"]:
    print(classify(text), text)

('予約', [('予約', 0.59), ('変更', 0.15)]) 土曜の 18 時に 4 人で行きたいんですが
('キャンセル', [('キャンセル', 0.83), ('雑談', 0.09)]) やっぱり行けなくなりました
('不明', [('キャンセル', 0.35), ('変更', 0.31)]) 予約を 1 時間遅らせたい
('営業時間', [('営業時間', 0.88), ('雑談', 0.04)]) 月曜って開いてる？
('場所', [('場所', 0.59), ('営業時間', 0.14)]) 車で行けますか
('予約', [('予約', 0.76), ('キャンセル', 0.1)]) 予約
('不明', [('場所', 0.41), ('雑談', 0.22)]) 領収書は出ますか？
('不明', [('営業時間', 0.49), ('雑談', 0.34)]) 今日の夕飯は何にしよう


In [2]:
# 交差検証で、例文だけでどのくらい分類できるかを確かめる
from sklearn.model_selection import cross_val_score

X = model.encode(["query: " + t for t in texts], normalize_embeddings=True)
print("5 分割交差検証の正解率:", round(cross_val_score(LogisticRegression(C=1000, max_iter=5000), X, labels, cv=5).mean(), 3))

5 分割交差検証の正解率: 0.9


- 意図ごとに 5 件ほどの例文でも、埋め込み + 線形分類器でかなり動く。誤分類された発話を例文に足していくと、すぐに改善する。
- 例文が少ないので、正則化を弱く (`C` を大きく) しないと確率が全体に平らになり、ほとんどが「不明」になる。
- 「予約」「キャンセル」「変更」のように似た意図は混同しやすい。確率が低いときは「不明」として、聞き返すか人に回す。
- どの意図にも当てはまらない発話 (領収書) を無理に分類しないよう、「その他」の例文を入れるか、確率のしきい値を設ける。